# Hướng dẫn chạy Mô hình 3D DATN trên Kaggle qua GitHub
Notebook này tự động thiết lập môi trường ảo Python 3.10, cài đặt các mô hình AI (SDXL, Grounded-SAM2, TRELLIS) và clone mã nguồn của bạn từ GitHub để chạy máy chủ API Ngrok kết nối với Giao diện Web.

## Bước 1: Khởi tạo Python 3.10 và Môi trường ảo (venv)

In [ ]:
import subprocess, os

def run(cmd):
    print(f"Executing: {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout.strip())
    if r.stderr and r.returncode != 0: print("ERROR:", r.stderr[-500:])
    return r.returncode

VENV = "/opt/venv310"
PY   = f"{VENV}/bin/python"

print("=== 1. Cài đặt Python 3.10 ===")
run("add-apt-repository ppa:deadsnakes/ppa -y")
run("apt-get update -qq")
run("apt-get install -qq python3.10 python3.10-dev python3.10-venv python3.10-distutils")

print("
=== 2. Khởi tạo môi trường ảo ===")
run(f"python3.10 -m venv {VENV}")
run(f"{PY} --version")

## Bước 2: Clone Repo GitHub của bạn và cài đặt thư viện

In [ ]:
# Nhập link Git chứa code của bạn tại đây
GITHUB_REPO = "https://github.com/USERNAME/REPO_NAME.git"  # <-- Thay thế bằng link Repo thật của bạn

import shutil
VENV = "/opt/venv310"
PIP = f"{VENV}/bin/pip"
TEMP_CLONE = "/tmp/my_repo"

# Xoá thư mục cũ nếu có
if os.path.exists(TEMP_CLONE): 
    shutil.rmtree(TEMP_CLONE)

print("=== 1. Tải mã nguồn từ GitHub ===")
!git clone {GITHUB_REPO} {TEMP_CLONE}

# Copy code vào thư mục làm việc chính
if os.path.exists(TEMP_CLONE):
    print("
=== 2. Đồng bộ mã nguồn vào thư mục làm việc ===")
    !cp -rf {TEMP_CLONE}/* /kaggle/working/
    print("✅ Đã copy files sang /kaggle/working/")
    !ls -l /kaggle/working/
else:
    print("❌ Lỗi: Không thể tải code từ Github. Vui lòng kiểm tra lại link repo.")

## Bước 3: Cài đặt dependencies từ requirements.txt

In [ ]:
VENV = "/opt/venv310"
PIP = f"{VENV}/bin/pip"

print("=== 1. Cài đặt các thư viện từ requirements.txt ===")
# Cài đặt PyTorch và CUDA 12.1 trước để đảm bảo tương thích xformers
!{PIP} install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121

# Cài đặt wheel và setuptools để hỗ trợ build package C++
!{PIP} install -q wheel setuptools

# Xoá dòng nvdiffrast nếu có trong file requirements để tránh lỗi cài đặt đè
!sed -i '/nvdiffrast/d' /kaggle/working/requirements.txt

# Cài đặt các package khác trong requirements.txt
!{PIP} install -q -r /kaggle/working/requirements.txt

# Ép cài đặt các package bổ sung cần thiết khác và đồng bộ numpy phiên bản tương thích PyTorch
!{PIP} install -q xformers==0.0.22.post7 --index-url https://download.pytorch.org/whl/cu121
!{PIP} install -q numpy==1.26.4
!{PIP} install -q spconv-cu121==2.3.8 plyfile==0.9

print("✅ Cài đặt thư viện hoàn tất!")

## Bước 4: Thiết lập và Biên dịch C++ Extension (nvdiffrast & diff-gaussian-rasterization)

In [ ]:
import subprocess, os, shutil

VENV = "/opt/venv310"
PIP  = f"{VENV}/bin/pip"
PY   = f"{VENV}/bin/python"

# 1. Đảm bảo import torch bình thường, nếu không thì tự sửa lỗi numpy
print("=== 1. Kiểm tra môi trường PyTorch ===")
check_torch = subprocess.run([PY, "-c", "import torch; print('PyTorch OK:', torch.__version__)"], capture_output=True, text=True)
if check_torch.returncode != 0:
    print("❌ Lỗi import PyTorch! Đang tiến hành cài đặt lại numpy phiên bản tương thích...")
    subprocess.run(f"{PIP} install -q --force-reinstall numpy==1.26.4", shell=True)
else:
    print("  ✓ PyTorch hoạt động tốt.")

# 2. Cài đặt diff-gaussian-rasterization
print("
=== 2. Cài đặt diff-gaussian-rasterization (Mip-Splatting) ===")
if os.path.exists("/tmp/mip-splatting"):
    shutil.rmtree("/tmp/mip-splatting")

!git clone --recursive https://github.com/autonomousvision/mip-splatting.git /tmp/mip-splatting
!{PIP} install -q --no-build-isolation --no-cache-dir /tmp/mip-splatting/submodules/diff-gaussian-rasterization

# 3. Vá lỗi PyTorch cpp_extension
print("
=== 3. Vá lỗi PyTorch cpp_extension để bỏ qua check CUDA ===")
cpp_ext_path = f"{VENV}/lib/python3.10/site-packages/torch/utils/cpp_extension.py"
if os.path.exists(cpp_ext_path):
    try:
        with open(cpp_ext_path, "r", encoding="utf-8") as f:
            code = f.read()
        target = "def _check_cuda_version(compiler_name, compiler_version):"
        patched = "def _check_cuda_version(compiler_name, compiler_version):
    return  # Patched by Antigravity to bypass CUDA version check"
        if target in code and "Patched by Antigravity" not in code:
            code = code.replace(target, patched)
            with open(cpp_ext_path, "w", encoding="utf-8") as f:
                f.write(code)
            print("  ✓ Vá lỗi file cpp_extension.py thành công!")
        else:
            print("  ✓ File cpp_extension.py đã được vá lỗi từ trước.")
    except Exception as e:
        print("  ❌ Lỗi khi vá file cpp_extension.py:", e)

# 4. Biên dịch nvdiffrast
print("
=== 4. Biên dịch nvdiffrast ===")
env = os.environ.copy()
cuda_path = "/usr/local/cuda"
if os.path.exists(cuda_path):
    env["CUDA_HOME"] = cuda_path
    env["PATH"] = f"{cuda_path}/bin:" + env.get("PATH", "")

print("Đang tiến hành compile nvdiffrast...")
res = subprocess.run(
    f"{PIP} install -v --no-cache-dir --no-build-isolation --no-deps --force-reinstall git+https://github.com/NVlabs/nvdiffrast.git",
    shell=True, env=env, capture_output=True, text=True
)

if res.returncode == 0:
    print("✅ Cài đặt nvdiffrast THÀNH CÔNG!")
else:
    print("❌ Biên dịch lỗi. Chi tiết lỗi bên dưới:")
    print(res.stderr[-2000:] if res.stderr else "Không có log lỗi STDERR.")

## Bước 4.5: Cài đặt mô hình phân mảnh Grounded-SAM2 & Tải các Checkpoints

In [ ]:
import os, subprocess
VENV = "/opt/venv310"
PIP  = f"{VENV}/bin/pip"

print("=== 1. Cài đặt GroundingDINO (bản pre-compiled PyPI) ===")
# Gỡ cài đặt bản local git lỗi nếu có để tránh xung đột
!{PIP} uninstall -y groundingdino
# Cài đặt từ PyPI
!{PIP} install -q --force-reinstall groundingdino-py addict yapf timm supervision

print("
=== 2. Cài đặt Segment Anything 2 (SAM2) ===")
if not os.path.exists("/tmp/segment-anything-2"):
    !git clone https://github.com/facebookresearch/segment-anything-2.git /tmp/segment-anything-2
    !{PIP} install -q --no-build-isolation -e /tmp/segment-anything-2
    print("  ✓ Cài đặt SAM2 hoàn tất.")
else:
    print("  ✓ SAM2 đã được cài đặt.")

print("
=== 3. Tải các Checkpoints mô hình ===")
dino_ckpt_dir = "/kaggle/working/groundingdino_ckpt"
os.makedirs(dino_ckpt_dir, exist_ok=True)
dino_ckpt_path = os.path.join(dino_ckpt_dir, "groundingdino_swint_ogc.pth")
if not os.path.exists(dino_ckpt_path):
    print("Đang tải GroundingDINO checkpoint...")
    !wget -q -O {dino_ckpt_path} https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
    print("  ✓ Tải GroundingDINO checkpoint xong.")
else:
    print("  ✓ Đã có GroundingDINO checkpoint.")

dino_config_path = os.path.join(dino_ckpt_dir, "GroundingDINO_SwinT_OGC.py")
if not os.path.exists(dino_config_path):
    print("Đang tải GroundingDINO config...")
    !wget -q -O {dino_config_path} https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py
    print("  ✓ Tải GroundingDINO config xong.")

sam2_ckpt_dir = "/kaggle/working/sam2_ckpt"
os.makedirs(sam2_ckpt_dir, exist_ok=True)
sam2_ckpt_path = os.path.join(sam2_ckpt_dir, "sam2_hiera_small.pt")
if not os.path.exists(sam2_ckpt_path):
    print("Đang tải SAM2 checkpoint...")
    !wget -q -O {sam2_ckpt_path} https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt
    print("  ✓ Tải SAM2 checkpoint xong.")
else:
    print("  ✓ Đã có SAM2 checkpoint.")

## Bước 5: Cài đặt Repo TRELLIS & Kiểm tra đăng nhập HuggingFace

In [ ]:
import os
TRELLIS_DIR = "/kaggle/working/TRELLIS"
if not os.path.exists(TRELLIS_DIR):
    print("=== 1. Tải mã nguồn mô hình TRELLIS ===")
    !GIT_LFS_SKIP_SMUDGE=1 git clone https://huggingface.co/spaces/trellis-community/TRELLIS {TRELLIS_DIR}
else:
    print("✅ Đã có thư mục TRELLIS")

print("
=== 2. Đăng nhập Hugging Face ===")
HF_TOKEN = "YOUR_HF_TOKEN"  # <-- Thay thế bằng HF token thật của bạn nếu cần

login_code = f'''
from huggingface_hub import login
login("{HF_TOKEN}")
print("✅ HF Login OK")
'''
!{PY} -c '{login_code}'

## Bước 6: Khởi chạy API Server và tạo link kết nối Ngrok

In [ ]:
import time, subprocess
from pyngrok import ngrok

# 1. Thay thế mã ngrok authtoken của bạn ở đây
NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN"  # <-- THAY THẾ BẰNG TOKEN NGROK THẬT CỦA BẠN
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Khởi động FastAPI server chạy ngầm dưới nền ở cổng 8000
print("🚀 Đang khởi động FastAPI Server ở cổng 8000...")
cmd_server = "/opt/venv310/bin/python /kaggle/working/server.py"

server_proc = subprocess.Popen(cmd_server.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
time.sleep(4)

# 3. Tạo đường truyền công khai (Tunnel) từ Ngrok kết nối với cổng 8000
try: 
    ngrok.kill() # Tắt các cổng thừa
    time.sleep(1)
    public_url = ngrok.connect(8000)
    print("\n" + "="*60)
    print("🎉 KHỒI ĐỘNG HỆ THỐNG THÀNH CÔNG!")
    print(f"👉 Đường dẫn API Ngrok của bạn: {public_url}")
    print("="*60 + "\n")
    print("Hãy copy đường dẫn trên dán vào ô 'Kaggle API URL' ở giao diện index.html trên máy tính của bạn.")
except Exception as e:
    print(f"❌ Lỗi khi mở cổng ngrok: {e}")